In [ ]:
#TRANSLATE 100 LANG

In [3]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
# nllb_langs = list(tokenizer.additional_special_tokens)

In [4]:
# nllb_langs

In [1]:
import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]

In [9]:
mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict.pop('en', None)
mapping_dict.pop('fr', None)
mapping_dict.pop('es', None)
mapping_dict.pop('zh', None)
mapping_dict.pop('hi', None)
mapping_dict.pop('ko', None)
len(mapping_dict)
# EXisting XNLI
keys_to_pop = ["ar","bg","de","el","en","es","fr","hi","ru","sw","th","tr","ur","vi","zh"]
for i in keys_to_pop:
    mapping_dict.pop(i,None)


In [11]:
# len(mapping_dict)

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import json
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.half().to("cuda")  # FP16
model.eval()

import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]

mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict.pop('en', None)
mapping_dict.pop('fr', None)
mapping_dict.pop('es', None)
mapping_dict.pop('zh', None)
mapping_dict.pop('hi', None)
mapping_dict.pop('ko', None)
keys_to_pop = ["ar","bg","de","el","en","es","fr","hi","ru","sw","th","tr","ur","vi","zh"]
for i in keys_to_pop:
    mapping_dict.pop(i,None)
len(mapping_dict)
lang_code = mapping_dict

def translate_batch(texts, target_lang):
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(lang_code[target_lang])
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs, forced_bos_token_id=forced_bos_token_id, max_length=256
        )
    translations = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
    return translations

def translate_function(dataset,target_langs,columns_to_translate):
    with open(output_file, "w", encoding="utf-8") as f:
        batch_rows = []
        
        for item in dataset:
            row = {col: item[col] for col in columns_to_translate if col in item}
            batch_rows.append(row)
            
            if len(batch_rows) == batch_size:
                # Translate each column for all target languages
                for lang in target_langs:
                    for col in columns_to_translate:
                        if col in row:
                            texts = [r[col] for r in batch_rows]
                            translated = translate_batch(texts, lang)
                            for i, r in enumerate(batch_rows):
                                r[f"{col}_{lang}"] = translated[i]
                
                # Write batch to JSONL
                for r in batch_rows:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
                
                batch_rows = []
    
        # Translate remaining rows
        if batch_rows:
            for lang in target_langs:
                for col in columns_to_translate:
                    if col in row:
                        texts = [r[col] for r in batch_rows]
                        translated = translate_batch(texts, lang)
                        for i, r in enumerate(batch_rows):
                            r[f"{col}_{lang}"] = translated[i]
            for r in batch_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    print(f"Multi-column multi-language translation completed! Saved to {output_file}")
batch_size = 64

#output_file = "mgsm_multicolumn_multilang.jsonl"
#columns_to_translate = ["question"] 
target_langs = list(mapping_dict)
#dataset = load_dataset("juletxara/mgsm", "en",split="test")
#translate_function(dataset,target_langs,columns_to_translate)


dataset = load_dataset("facebook/xnli", "en", split="test")
columns_to_translate = ["premise","hypothesis"]
output_file = "xnli_multicolumn_multilang.jsonl"
translate_function(dataset,target_langs,columns_to_translate)


Multi-column multi-language translation completed! Saved to xnli_multicolumn_multilang.jsonl


In [3]:
# lang_list = []
# dataset_list = {} # with names and fields to translate # Chines has 2 scripts

In [4]:
# dataset = load_dataset("jsonl", data_files="/scratch/amahaj56/sampled_olmo.jsonl", split="train")

4

4

Translating language: als


TypeError: string indices must be integers, not 'str'